In [3]:
import pandas as pd

# Cargar el dataset desde la carpeta data relativa al notebook
df = pd.read_csv("data/housing.csv")

# Confirmar lectura
print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Separar variables predictoras (X) y objetivo (y)
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

# Realizar la partición 80% Train / 20% Test (fijando semilla 42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train_raw.shape}, Test set: {X_test_raw.shape}")

Train set: (16512, 9), Test set: (4128, 9)


In [5]:
# Guardar particiones crudas en fase-1/data/
X_train_raw.to_csv('data/X_train_raw.csv', index=False)
X_test_raw.to_csv('data/X_test_raw.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

In [6]:
# Entrenar DummyRegressor usando la media del set de entrenamiento
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train_raw, y_train)

# Predicciones sobre el set de prueba
y_pred_dummy = dummy.predict(X_test_raw)

# Calcular métricas de evaluación
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2_dummy = r2_score(y_test, y_pred_dummy)

print(f"Baseline (Dummy) -> MAE: {mae_dummy:.2f}, RMSE: {rmse_dummy:.2f}, R2: {r2_dummy:.4f}")

# Registrar la métrica en el archivo resultados/metricas.csv
import csv
import os

# Crear la carpeta resultados si no existe
os.makedirs("resultados", exist_ok=True)

linea = [
    "Baseline_Mean",
    "Oscar",
    f"{mae_dummy:.2f}",
    f"{rmse_dummy:.2f}",
    f"{r2_dummy:.4f}",
]
with open("resultados/metricas.csv", mode="a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(linea)

print(
    "¡Métricas guardadas exitosamente en resultados/metricas.csv!"
)

Baseline (Dummy) -> MAE: 90606.85, RMSE: 114485.64, R2: -0.0002
¡Métricas guardadas exitosamente en resultados/metricas.csv!


In [7]:
from sklearn.linear_model import LinearRegression

# 1. Seleccionar solo variables numéricas para el primer experimento
X_train_num = X_train_raw.select_dtypes(include=[np.number]).copy()
X_test_num = X_test_raw.select_dtypes(include=[np.number]).copy()

# 2. Imputar valores faltantes (usando la mediana de cada columna)
X_train_num = X_train_num.fillna(X_train_num.median())
X_test_num = X_test_num.fillna(X_train_num.median())

# 3. Entrenar el modelo de Regresión Lineal
lr = LinearRegression()
lr.fit(X_train_num, y_train)

# 4. Generar predicciones
y_pred_lr = lr.predict(X_test_num)

# 5. Calcular métricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Regresión Lineal -> MAE: {mae_lr:.2f}, RMSE: {rmse_lr:.2f}, R2: {r2_lr:.4f}")

# 6. Guardar la nueva métrica en resultados/metricas.csv
linea_lr = [
    "Linear_Regression_Simple",
    "Oscar",
    f"{mae_lr:.2f}",
    f"{rmse_lr:.2f}",
    f"{r2_lr:.4f}"
]

with open("resultados/metricas.csv", mode="a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(linea_lr)

print("¡Nueva métrica guardada en resultados/metricas.csv!")

Regresión Lineal -> MAE: 51810.09, RMSE: 71131.26, R2: 0.6139
¡Nueva métrica guardada en resultados/metricas.csv!
